# 09 · Block-select — block 11 vs block 12 làm đầu vào ML-Decoder

`08` đo bằng linear probe: **block 11 (mAP 0.4026) > block 12 (0.3962), −0.0064**. Backbone
tự giám sát thường để block cuối chuyên hóa cho mục tiêu pretrain, vứt bỏ thông tin xuôi dòng.
Nhưng probe đo trên `mean(patch)` + linear; ML-Decoder cross-attend **toàn bộ 1369 token**.
Notebook này kiểm xem lợi thế đó có **chuyển sang decoder** không.

## Bốn arm — cùng điều kiện, chỉ khác nguồn token

| arm | token vào decoder | vai trò |
|---|---|---|
| `b12ln` | block 12 **+ layernorm cuối** = `last_hidden_state` | **tái lập H5** (cổng sanity, phải ≈ 0.4336) |
| `b12` | block 12 **thô** (bỏ layernorm cuối) | kiểm: layernorm pretrain giúp hay hại |
| `b11` | block 11 **thô** | **giả thuyết** — probe nói cái này cao hơn |
| `b1112` | block 11 ⊕ block 12 (nối theo token, 2738) | "cho decoder cả hai" |

**Estimand chính:** `b11 − b12` (cùng thô ⇒ cô lập đúng lựa chọn block).
Phụ: `b12ln − b12` (layernorm cuối), `b1112 − b12` (cả hai block).

## Kỷ luật — giống 06/06b/07

- **3 seed** `[42,1337,2024]`, **ghép cặp** (cùng seed = cùng init, cùng thứ tự data, cùng ảnh).
- Ngưỡng **±0.005**: dưới nó không phát biểu được.
- **Backbone đông cứng** (04b đã chứng minh trích đông cứng an toàn). Một forward chung cho cả 4 arm.
- **Val là trọng tài.** Test giữ nguyên, chỉ chạm **một lần** ở C12 nếu val vượt +0.005.
- Loss/optimizer/schedule/early-stop/pooling **y hệt H5** để `b12ln` tái lập được con số cũ.


## C0 · Config

In [ ]:
import os, json, math, time, pickle, shutil, tarfile, random
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
LOCAL_IMG = '/content/cxr_local'                       # o tam VM, XOA moi restart -> C0-bis
SHARD_DIR = os.path.join(DATA_DIR, 'cxr_560_shards')
BEST_CKPT = os.path.join(DATA_DIR, 'lpft_v2', 'raddino_mld_db_ft_best.pt')
BS_DIR    = os.path.join(DATA_DIR, 'block_select'); os.makedirs(BS_DIR, exist_ok=True)
os.makedirs(f'{BS_DIR}/checkpoints', exist_ok=True); os.makedirs(f'{BS_DIR}/metrics', exist_ok=True)

W_F, W_L = 0.7, 0.3                                    # study-level pooling, giong 04b/05/06b
SEEDS    = [42, 1337, 2024]
EPOCHS, PATIENCE = 12, 3
EFFECTIVE_BATCH  = 256
LR, WD, WARMUP, CLIP = 1e-3, 1e-2, 0.05, 5.0
AMP   = torch.bfloat16
BATCH = 32                                             # b1112 (2738 token) la arm ton bo nho nhat
ARMS  = ['b12ln', 'b12', 'b11', 'b1112']              # b12ln = tai lap H5
H5_REF = 0.4336                                        # con so H5 de doi chieu cong sanity

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, '|', torch.cuda.get_device_name(0) if dev=='cuda' else 'CPU')
print('arms:', ARMS, '| seeds:', SEEDS, '| epochs', EPOCHS, 'patience', PATIENCE)

## C0-bis · Stage ảnh về ổ local

Ảnh trên Drive là **tar shard** (`cxr_560_shards`); `/content` bị xóa mỗi lần VM restart.
Phải chạy **trong phiên này** dù phiên trước đã stage. **CPU/network** — ~8 phút, tức thì nếu ổ ấm.

In [ ]:
os.makedirs(LOCAL_IMG, exist_ok=True)
tars = sorted(f for f in os.listdir(SHARD_DIR) if f.endswith('.tar'))
assert tars, f'no shards in {SHARD_DIR} -> run 04a first'
todo = [f for f in tars if not os.path.exists(os.path.join(LOCAL_IMG, f'.done_{f}'))]
print(f'shards: {len(tars)} on Drive | {len(todo)} to stage')
for f in tqdm(todo, desc='staging shards', unit='shard'):
    tmp = f'/content/{f}'
    shutil.copyfile(os.path.join(SHARD_DIR, f), tmp)
    with tarfile.open(tmp) as t: t.extractall(LOCAL_IMG)
    os.remove(tmp)
    open(os.path.join(LOCAL_IMG, f'.done_{f}'), 'w').close()   # marker AFTER extract
N_LOCAL = len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')])
print(f'local images: {N_LOCAL:,}')
assert N_LOCAL > 0, 'staging produced no images'

## C1 · Cohort, nhãn, manifest ảnh (image-only — không cần EHR)

In [ ]:
meta   = json.load(open(os.path.join(OUT_DIR, 'cohort_meta.json')))
LABELS = meta['labels']; assert len(LABELS) == 13

ehr = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_study_level.parquet')).sort_values('study_id').reset_index(drop=True)
Y_STUDY  = ehr[LABELS].to_numpy('float32')             # theo hang cua ehr
study2row = {int(s): i for i, s in enumerate(ehr.study_id)}

vm = pd.read_parquet(os.path.join(OUT_DIR, 'view_map.parquet'))
vm = vm[vm.view_group.isin(['frontal','lateral'])]
_dup = [c for c in ('subject_id','split','setting') if c in vm.columns]
man = vm.drop(columns=_dup).merge(ehr[['study_id','split']], on='study_id', how='inner')

_before = len(man)
man = man[man.dicom_id.map(lambda d: os.path.exists(f'{LOCAL_IMG}/{d}.jpg'))].reset_index(drop=True)
assert len(man), (f'ALL {_before:,} images missing from {LOCAL_IMG} -> chay C0-bis truoc '
                  '(100% missing = VM moi, /content da bi xoa)')
assert (_before-len(man))/_before < 0.01, f'{_before-len(man):,}/{_before:,} images missing -> chay lai C0-bis'

man['yrow'] = man.study_id.map(study2row).astype('int64')
assert man.yrow.notna().all()
assert man.groupby('study_id')['split'].nunique().max() == 1, 'a study spans two splits'

IDX = {s: np.where(man.split.to_numpy() == s)[0] for s in ('train','val','test')}
print(f'manifest: {len(man):,} images / {man.study_id.nunique():,} studies')
print('by split :', {s: int(len(IDX[s])) for s in IDX})
print('by view  :', dict(man.view_group.value_counts()))

## C2 · Dataset ở mức ảnh + transform (giống 04b/06b, KHÔNG flip)

`RandomResizedCrop`+rotation cho train, `CenterCrop` cho val/test. Không flip: trái/phải có
nghĩa trên phim ngực. Cả 4 arm nhận **cùng một ảnh đã augment** nhờ shared forward.

In [ ]:
from transformers import AutoImageProcessor
from torchvision import transforms as T
from PIL import Image

proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MODEL_SIZE = proc.crop_size['height']; MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(MODEL_SIZE, scale=(0.85,1.0), ratio=(0.95,1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])
eval_tf  = T.Compose([T.CenterCrop(MODEL_SIZE), T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])

class ImgDS(Dataset):
    def __init__(self, idx, tf): self.idx, self.tf = np.asarray(idx), tf
    def __len__(self): return len(self.idx)
    def __getitem__(self, k):
        r = man.iloc[self.idx[k]]
        im = Image.open(f'{LOCAL_IMG}/{r.dicom_id}.jpg').convert('L')
        return dict(image=self.tf(Image.merge('RGB', (im, im, im))),
                    y=torch.from_numpy(Y_STUDY[int(r.yrow)]))

def collate(b):
    return dict(image=torch.stack([x['image'] for x in b]),
                y=torch.stack([x['y'] for x in b]))

def loader(split, tf, shuffle, bs, workers=8):
    # workers>0 chi cho TRAIN loader song lau (fork 1 lan); predict dung workers=0 -> khong
    # co teardown 'can only test a child process'. Frozen ViT forward moi la nut co chai.
    return DataLoader(ImgDS(IDX[split], tf), batch_size=bs, shuffle=shuffle, collate_fn=collate,
                      num_workers=workers, pin_memory=(dev=='cuda'),
                      drop_last=False, persistent_workers=(workers > 0))
to_dev = lambda b: {k: v.to(dev, non_blocking=True) for k, v in b.items()}
print('loaders ready:', {s: int(len(IDX[s])) for s in IDX})

## C3 · ML-Decoder + shim torch 2.x (giống 04b/06b)

`TransformerDecoderLayerOptimal` **không có self-attention** (nghĩa của "Optimal"), nên hai
shim đều no-op chứng minh được: cờ `batch_first` và bỏ `tgt/memory_is_causal`.

In [ ]:
ML_SRC = '/content/ml_decoder.py'
if not os.path.exists(ML_SRC):
    !wget -q -O {ML_SRC} https://raw.githubusercontent.com/Alibaba-MIIL/ML_Decoder/main/src_files/ml_decoder/ml_decoder.py
import sys, types, inspect
sys.path.insert(0, '/content')
import ml_decoder as _mld
from ml_decoder import MLDecoder

if not hasattr(_mld.TransformerDecoderLayerOptimal, '_patched_for_torch2'):
    _oi = _mld.TransformerDecoderLayerOptimal.__init__
    def _init(self, *a, **kw):
        _oi(self, *a, **kw); self.self_attn = types.SimpleNamespace(batch_first=False)
    _mld.TransformerDecoderLayerOptimal.__init__ = _init
    _of = _mld.TransformerDecoderLayerOptimal.forward
    def _fwd(self, tgt, memory, **kw):
        kw.pop('tgt_is_causal', None); kw.pop('memory_is_causal', None)
        return _of(self, tgt, memory, **kw)
    _mld.TransformerDecoderLayerOptimal.forward = _fwd
    _mld.TransformerDecoderLayerOptimal._patched_for_torch2 = True

def make_mld():
    """13 fixed queries cross-attending a [B, N, 768] token sequence (N khac nhau moi arm)."""
    return MLDecoder(num_classes=len(LABELS), initial_num_features=768,
                     decoder_embedding=768, num_of_groups=len(LABELS))
_m = make_mld()
print('MLDecoder ok |', inspect.signature(MLDecoder.__init__))
print('query_embed learnable:', _m.decoder.query_embed.weight.requires_grad, '(False by design)')
del _m

## C4 · Backbone đông cứng + `forward_blocks`

`output_hidden_states=True` trả tuple 13 phần: `hidden_states[i]` = đầu ra layer thứ `i` (raw,
**trước** layernorm cuối). `last_hidden_state = layernorm(hidden_states[12])`.

- `b11` = `hidden_states[11][:,1:]` (patch của block 11, thô)
- `b12` = `hidden_states[12][:,1:]` (block 12, thô)
- `b12ln` = `layernorm(hidden_states[12])[:,1:]` = **đúng thứ H5 thấy**
- `b1112` = `cat([b11, b12], dim=1)` (2738 token)

In [ ]:
from transformers import AutoModel
backbone = AutoModel.from_pretrained('microsoft/rad-dino').to(dev).eval()
_sd = torch.load(BEST_CKPT, map_location=dev, weights_only=False)['model']
_bb = {k[len('backbone.'):]: v for k, v in _sd.items() if k.startswith('backbone.')}
assert _bb, 'no backbone.* keys in checkpoint'
backbone.load_state_dict(_bb, strict=True)
for p in backbone.parameters(): p.requires_grad = False
assert sum(p.requires_grad for p in backbone.parameters()) == 0
D = backbone.config.hidden_size; assert D == 768
NB = backbone.config.num_hidden_layers
assert hasattr(backbone, 'layernorm'), 'Dinov2Model can co .layernorm (final LN)'
_final_ln = backbone.layernorm
print(f'frozen backbone: {len(_bb)} tensors | blocks {NB} | hidden {D}')

@torch.no_grad()
def forward_blocks(images):
    """MOT forward chung. Tra dict token da detach cho ca 4 arm."""
    with torch.autocast('cuda', dtype=AMP, enabled=(dev=='cuda')):
        o = backbone(pixel_values=images, output_hidden_states=True)
    hs = o.hidden_states                                   # tuple len NB+1
    p11   = hs[NB-1][:, 1:].detach().float()               # block 11 (raw)
    p12   = hs[NB  ][:, 1:].detach().float()               # block 12 (raw)
    p12ln = _final_ln(hs[NB])[:, 1:].detach().float()      # = last_hidden_state[:,1:]
    return {'b11': p11, 'b12': p12, 'b12ln': p12ln,
            'b1112': torch.cat([p11, p12], 1), '_lhs': o.last_hidden_state}

## C5 · Arm = LayerNorm + ML-Decoder  ·  ASLoss (verbatim 06b)

`b12ln` = `mld(LN(layernorm(hs12)[:,1:]))` — **đúng bằng H5** (H5 cũng LN rồi mld trên
`last_hidden_state`). Nên nó phải tái lập được 0.4336.

In [ ]:
class BlockArm(nn.Module):
    def __init__(self, src):
        super().__init__(); self.src = src
        self.norm = nn.LayerNorm(D); self.mld = make_mld()
    def forward(self, feats):
        return self.mld(self.norm(feats[self.src]))

def build_arms(seed):
    g = torch.Generator().manual_seed(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    return {t: BlockArm(t).to(dev) for t in ARMS}

class ASLoss(nn.Module):
    """Asymmetric Loss (Ben-Baruch 2021), Alibaba-MIIL/ASL, math verbatim."""
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super().__init__(); self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps
    def forward(self, x, y):
        y = y.float(); xs_pos = torch.sigmoid(x); xs_neg = 1.0 - xs_pos
        if self.clip: xs_neg = (xs_neg + self.clip).clamp(max=1)
        los = y*torch.log(xs_pos.clamp(min=self.eps)) + (1-y)*torch.log(xs_neg.clamp(min=self.eps))
        pt = xs_pos*y + xs_neg*(1-y); g = self.gp*y + self.gn*(1-y)
        return -(los*torch.pow(1-pt, g)).mean()
print('BlockArm + ASLoss ready')

## C6 · Predict ở mức ảnh → pool 7:3 ở mức study (giống 06b)

In [ ]:
from sklearn.metrics import average_precision_score

@torch.no_grad()
def predict_images(arms, split, bs):
    for m in arms.values(): m.eval()
    P = {t: np.zeros((len(IDX[split]), len(LABELS)), 'float32') for t in arms}
    pos = 0
    for b in tqdm(loader(split, eval_tf, False, bs, workers=0), desc=f'predict[{split}]', leave=False):
        b = to_dev(b); feats = forward_blocks(b['image']); n = b['image'].shape[0]
        for t, m in arms.items():
            P[t][pos:pos+n] = torch.sigmoid(m(feats)).float().cpu().numpy()
        pos += n
    return P

def pool_study(p_img, split):
    sub = man.iloc[IDX[split]]
    d = pd.DataFrame(p_img, columns=LABELS)
    d['study_id'] = sub.study_id.to_numpy(); d['vg'] = sub.view_group.to_numpy()
    sid = np.sort(d.study_id.unique())
    fr = d[d.vg=='frontal'].groupby('study_id')[LABELS].mean().reindex(sid)
    la = d[d.vg=='lateral'].groupby('study_id')[LABELS].mean().reindex(sid)
    hf = fr.notna().all(1).to_numpy().astype('float32'); hl = la.notna().all(1).to_numpy().astype('float32')
    den = (W_F*hf + W_L*hl)[:, None]; assert (den > 0).all()
    PS = (W_F*np.nan_to_num(fr.to_numpy('float32'))*hf[:,None]
          + W_L*np.nan_to_num(la.to_numpy('float32'))*hl[:,None]) / den
    return sid, PS

def ap_study(p_img, split):
    sid, PS = pool_study(p_img, split)
    Yt = ehr.set_index('study_id').loc[sid, LABELS].to_numpy('float32')
    return np.array([average_precision_score(Yt[:,j], PS[:,j]) for j in range(len(LABELS))]), sid, PS

## C7 · Train dùng chung forward — mỗi arm optimizer/schedule/clip/early-stop RIÊNG

Backward tuần tự trên feature đã `detach()`: không giữ 4 graph cùng lúc, clip **riêng** từng
arm (clip global sẽ trộn scale gradient). Resume **mỗi epoch** — 06b từng mất 110 phút vì
thiếu nó. Scheduler cũng được lưu: thiếu nó cosine nhảy về đỉnh giữa chừng.

In [ ]:
def param_groups(m, wd):
    dec = [p for p in m.parameters() if p.requires_grad and p.ndim > 1]
    nod = [p for p in m.parameters() if p.requires_grad and p.ndim <= 1]
    return [{'params':dec,'weight_decay':wd},{'params':nod,'weight_decay':0.0}]

def cosine_warmup(opt, total, warm):
    f = lambda s: s/max(warm,1) if s < warm else 0.5*(1+math.cos(math.pi*(s-warm)/max(total-warm,1)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

class Exp:
    def __init__(self, tag, model, total):
        self.tag, self.model = tag, model
        self.crit = ASLoss()
        self.opt  = torch.optim.AdamW(param_groups(model, WD), lr=LR)
        self.sch  = cosine_warmup(self.opt, total, int(WARMUP*total))
        self.best, self.since, self.done, self.best_sd = -1.0, 0, False, None

def ck_resume(seed): return f'{BS_DIR}/checkpoints/_resume_s{seed}.pt'
batch_of, epochs_of = {}, {}

def save_resume(seed, exps, ep):
    obj = {'epoch': ep, 'batch': batch_of[seed], 'epochs': epochs_of[seed],
           'arms': {t: dict(model=e.model.state_dict(), opt=e.opt.state_dict(), sch=e.sch.state_dict(),
                            best=e.best, since=e.since, done=e.done, best_sd=e.best_sd)
                    for t, e in exps.items()}}
    p = ck_resume(seed); torch.save(obj, p + '.tmp'); os.replace(p + '.tmp', p)   # atomic

def train_shared(seed, batch=BATCH, epochs=EPOCHS, verbose=True):
    arms = build_arms(seed)
    dl = loader('train', train_tf, True, batch)
    accum = max(1, EFFECTIVE_BATCH // batch)
    total = epochs * math.ceil(len(dl)/accum)
    exps  = {t: Exp(t, m, total) for t, m in arms.items()}
    if verbose: print(f'seed {seed} | batch {batch} x accum {accum} = {batch*accum} eff | '
                      f'{len(dl)} it/epoch | {total} opt-steps')
    batch_of[seed], epochs_of[seed] = batch, epochs
    start = 0
    if os.path.exists(ck_resume(seed)):
        c = torch.load(ck_resume(seed), map_location=dev, weights_only=False)
        if (c.get('batch'), c.get('epochs')) != (batch, epochs):
            raise RuntimeError(f"resume seed {seed} built with batch={c.get('batch')} epochs={c.get('epochs')}"
                               f" but running batch={batch} epochs={epochs} -> LR schedule wrong. "
                               f"Delete {ck_resume(seed)} or match the old values.")
        for t, e in exps.items():
            a = c['arms'][t]
            e.model.load_state_dict(a['model']); e.opt.load_state_dict(a['opt']); e.sch.load_state_dict(a['sch'])
            e.best, e.since, e.done, e.best_sd = a['best'], a['since'], a['done'], a['best_sd']
        start = c['epoch'] + 1
        if verbose: print(f'RESUMED seed {seed} at epoch {start}/{epochs}  ' +
                          ' '.join(f'{t}:{e.best:.4f}{"(done)" if e.done else ""}' for t, e in exps.items()))
    if start >= epochs or all(e.done for e in exps.values()):
        for e in exps.values(): e.model.load_state_dict(e.best_sd)
        if verbose: print(f'seed {seed} already complete');
        return {t: e.model for t, e in exps.items()}, {t: e.best for t, e in exps.items()}

    for ep in range(start, epochs):
        for m in arms.values(): m.train()
        for i, b in enumerate(tqdm(dl, desc=f's{seed} ep{ep+1}/{epochs}', leave=False)):
            b = to_dev(b); feats = forward_blocks(b['image'])          # THE shared forward
            for t, e in exps.items():
                if e.done: continue
                loss = e.crit(e.model(feats), b['y']) / accum
                loss.backward()                                        # per-arm graph; feats detached
                if (i+1) % accum == 0 or i == len(dl)-1:
                    nn.utils.clip_grad_norm_(e.model.parameters(), CLIP)   # PER ARM
                    e.opt.step(); e.sch.step(); e.opt.zero_grad(set_to_none=True)
        Pv = predict_images({t:e.model for t,e in exps.items() if not e.done}, 'val', batch*2)
        line = []
        for t, e in exps.items():
            if e.done: line.append(f'{t}:done'); continue
            mAP = ap_study(Pv[t], 'val')[0].mean()
            if mAP > e.best:
                e.best, e.since = mAP, 0
                e.best_sd = {k: v.detach().cpu().clone() for k, v in e.model.state_dict().items()}
            else:
                e.since += 1
                if e.since >= PATIENCE: e.done = True
            line.append(f'{t}:{mAP:.4f}{"*" if e.since==0 else ""}')
        save_resume(seed, exps, ep)
        if verbose: print(f'  ep{ep+1} val ' + ' '.join(line) + '  [resume saved]')
        if all(e.done for e in exps.values()):
            if verbose: print('  all arms early-stopped'); break
    for e in exps.values(): e.model.load_state_dict(e.best_sd)
    return {t: e.model for t, e in exps.items()}, {t: e.best for t, e in exps.items()}

## C8 · Cổng kiểm tra — chạy TRƯỚC khi tốn GPU

1. **Shapes** — b11/b12/b12ln = `[B,1369,768]`, b1112 = `[B,2738,768]`.
2. **`b12ln` = last_hidden_state** — `layernorm(hs12)` phải trùng `last_hidden_state` (chứng
   minh `b12ln` đúng là thứ H5 thấy).
3. **block 11 ≠ block 12** — nếu chúng bằng nhau thì cả thí nghiệm vô nghĩa. In L2 + cosine.
4. **ASL** — easy-neg ~0, hard-neg lớn.

In [ ]:
_bt = next(iter(loader('val', eval_tf, False, 4, workers=0)))
_ft = forward_blocks(_bt['image'].to(dev))
print('shapes:', {k: tuple(v.shape) for k, v in _ft.items() if k != '_lhs'})
assert _ft['b11'].shape[1] == 1369 and _ft['b1112'].shape[1] == 2738

_o = backbone(pixel_values=_bt['image'].to(dev), output_hidden_states=True)   # MOT forward
d_lhs = (_final_ln(_o.hidden_states[NB]) - _o.last_hidden_state).abs().max().item()
print(f'|layernorm(hs12) - last_hidden_state| max = {d_lhs:.2e}  (phai ~0 -> b12ln == input cua H5)')
assert d_lhs < 1e-3

l2  = (_ft['b11'] - _ft['b12']).norm(dim=-1).mean().item()
cos = F.cosine_similarity(_ft['b11'], _ft['b12'], dim=-1).mean().item()
print(f'block11 vs block12: mean L2 {l2:.3f} | mean cosine {cos:.3f}  (phai khac nhau ro)')
assert l2 > 0.1 and cos < 0.999

with torch.no_grad():
    _a = ASLoss()
    _e = _a(torch.tensor([[-4.6]]), torch.tensor([[0.0]])).item()
    _h = _a(torch.tensor([[ 4.6]]), torch.tensor([[0.0]])).item()
print(f'ASL: easy-neg {_e:.5f} (~0) | hard-neg {_h:.3f} (lon)')
assert _e < 1e-3 and _h > 1.0
print('\nGATES OK')

## C9 · Pilot seed 42, 1 epoch — xác nhận pipeline chạy & số hợp lý

**KHÔNG dùng để kết luận.** Chỉ để thấy 4 arm train được, val ra số hợp lý, trước khi cam
kết ~1.5–2h. Đặt `PILOT=True`. Xong thì đặt `False` và chạy C10.

In [ ]:
PILOT = True
if PILOT:
    _p = ck_resume(42)
    if os.path.exists(_p):
        print('resume seed 42 ton tai -> bo qua pilot (C10 se tiep tuc).')
    else:
        t0 = time.time()
        _m, _best = train_shared(42, epochs=1, verbose=True)
        print(f'\npilot 1 epoch: {(time.time()-t0)/60:.1f} phut')
        print('val macro (1 epoch, chua hoi tu):', {t: round(float(v),4) for t,v in _best.items()})
        # dung resume cua pilot (epochs=1) cho lan chay day du (epochs=12)
        os.remove(ck_resume(42)); print('da xoa resume pilot (epochs=1) -> C10 chay lai seed 42 tu dau voi epochs=12')

## C10 · Chạy đầy đủ 3 seed × 12 epoch (resume mỗi epoch)

**GPU A100.** ~1.5–2h. Backbone forward (một lần, chung cho 4 arm) là nút cổ chai; b1112
(2738 token) là arm nặng nhất về bộ nhớ decoder. Mất mạng giữa chừng: chạy lại cell này,
nó resume từ epoch cuối đã lưu.

In [ ]:
VAL_BEST = {}
for s in SEEDS:
    print(f'\n===== seed {s} =====')
    _models, _best = train_shared(s, epochs=EPOCHS, verbose=True)
    VAL_BEST[s] = {t: float(v) for t, v in _best.items()}
    json.dump(VAL_BEST, open(f'{BS_DIR}/metrics/val_best.json','w'), indent=1)
    print(f'seed {s} val best: ' + ' '.join(f'{t}:{VAL_BEST[s][t]:.4f}' for t in ARMS))
print('\nsaved -> metrics/val_best.json')

## C11 · Kết quả — CI ghép cặp theo seed

`b12ln` phải ≈ H5 (0.4336). Estimand chính `b11 − b12`. Ngưỡng ±0.005.

In [ ]:
from scipy import stats as sstats
VAL_BEST = json.load(open(f'{BS_DIR}/metrics/val_best.json'))
ss = sorted(VAL_BEST, key=lambda x: int(x))
M  = lambda t: np.array([VAL_BEST[s][t] for s in ss])
def ci95(v):
    v = np.asarray(v, float); n = len(v)
    return (float(v.mean()), 0.0) if n < 2 else \
           (float(v.mean()), float(sstats.t.ppf(0.975, n-1)*v.std(ddof=1)/np.sqrt(n)))

print('===== val macro mAP theo seed (mean +/- 95% CI) =====')
print(pd.DataFrame([dict(arm=t, **dict(zip(('mean','ci95'), np.round(ci95(M(t)),4)))) for t in ARMS]
      ).to_string(index=False))

print(f'\n===== cong sanity: b12ln tai lap H5? (ref {H5_REF}) =====')
mu, h = ci95(M('b12ln'))
print(f'  b12ln = {mu:.4f} +/-{h:.4f}   lech H5 = {mu-H5_REF:+.4f}   '
      f'{"OK (trong ~0.01)" if abs(mu-H5_REF) < 0.012 else "!! lech xa -> kiem lai truoc khi tin phan con lai"}')

def paired(name, vec):
    mu, h = ci95(vec); t, p = sstats.ttest_1samp(vec, 0.0) if len(vec) > 1 else (0, 1)
    verdict = 'phat bieu duoc' if (abs(mu) > 0.005 and p < 0.05) else 'DUOI nguong / khong phan biet'
    print(f'  {name:24s} {mu:+.4f} +/-{h:.4f}  p={p:.3f}  -> {verdict}')

print('\n===== ESTIMAND =====')
paired('b11 - b12  (CHINH)', M('b11') - M('b12'))
print('\n===== phu =====')
paired('b12ln - b12', M('b12ln') - M('b12'))
paired('b1112 - b12', M('b1112') - M('b12'))
paired('b11 - b12ln', M('b11') - M('b12ln'))

best_arm = max(ARMS, key=lambda t: M(t).mean())
gain = M(best_arm).mean() - M('b12ln').mean()
print(f'\narm tot nhat: {best_arm} ({M(best_arm).mean():.4f})  |  hon b12ln (=H5) {gain:+.4f}')
print(f'-> {"vuot +0.005: dang chay confirmatory tren TEST (C12)" if gain > 0.005 else "duoi +0.005: KHONG mo test. Ket luan: block cuoi khong phi thong tin dang ke qua decoder."}')

## C12 · Confirmatory trên TEST — **chỉ chạy nếu C11 vượt +0.005**

Test giữ nguyên suốt 06/06b/07/09. Chạm **một lần**, đúng arm thắng vs `b12ln`, 3 seed.
Đặt `RUN_TEST=True` **chỉ khi** C11 cho `gain > 0.005`.

In [ ]:
RUN_TEST = False
if RUN_TEST:
    WIN = best_arm     # tu C11
    AP_TEST = {}
    for s in SEEDS:
        arms = build_arms(s)
        c = torch.load(ck_resume(s), map_location=dev, weights_only=False)   # da hoi tu
        for t in (WIN, 'b12ln'):
            arms[t].load_state_dict(c['arms'][t]['best_sd'])
        Pt = predict_images({t: arms[t] for t in (WIN, 'b12ln')}, 'test', BATCH*2)
        AP_TEST[s] = {t: ap_study(Pt[t], 'test')[0] for t in (WIN, 'b12ln')}
    Mt = lambda t: np.array([AP_TEST[s][t].mean() for s in SEEDS])
    d = Mt(WIN) - Mt('b12ln'); mu, h = ci95(d); _, p = sstats.ttest_1samp(d, 0.0)
    print(f'TEST {WIN} = {Mt(WIN).mean():.4f} | b12ln = {Mt("b12ln").mean():.4f}')
    print(f'TEST {WIN} - b12ln = {mu:+.4f} +/-{h:.4f}  p={p:.3f}')
    json.dump({s: {t: AP_TEST[s][t].tolist() for t in AP_TEST[s]} for s in AP_TEST},
              open(f'{BS_DIR}/metrics/test_confirm.json','w'), indent=1)
else:
    print('RUN_TEST=False -> test khong bi cham. Dat True CHI KHI C11 gain > 0.005.')